In [2]:
import os

# get Groq api key by reading the first line of the text file Open AI coding.txt with encoding utf-8

with open('groq.txt') as file:
    api_key = str(file.readline())

os.environ['GROQ_API_KEY'] = api_key

In [ ]:
#source: https://learn.deeplearning.ai/courses/langchain-chat-with-your-data/lesson/2/document-loading
# pip install langchain-community pypdf
# we need to load our pdf document

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("Session 1-5.pdf")
pages = loader.load()

In [6]:
len(pages)

2

In [7]:
page = pages[0]

In [8]:
start = page.page_content
print(start)

file:///C/...20-%20WHU/Dokumente/Onderwijs/Digital%20technology%20strategy/Online%20MBA%202023/Video/Session%201-5.txt[3/27/2024 5:56:26 PM]
 Dear MBA participants in this video we will focus on the topic of how you can actually respond to digital 
description as a company And actually there are three different response strategies that companies can apply The 
first one is to team up which means that you engage in collaboration or even acquire a disruptor a Second strategy 
can be to fight back So that you really decide okay, we are disrupted by another company now. Let's fight back and 
create our own disrupted business model and Finally, and this is quite a special strategy you can also go for the 
escape option and I will later on explain what that exactly means So let's first focus on the team up strategy So this is 
a strategy where you decide as a company to actually collaborate with a potential Disruptor of your business model 
and here is just one example. So Zalando is one of 

In [11]:
# now we need to chunk the text

#source: https://learn.deeplearning.ai/courses/langchain-chat-with-your-data/lesson/3/document-splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0
)

splitted = r_splitter.split_text(start)

splitted

['file:///C/...20-%20WHU/Dokumente/Onderwijs/Digital%20technology%20strategy/Online%20MBA%202023/Video/Session%201-5.txt[3/27/2024 5:56:26 PM]\n Dear MBA participants in this video we will focus on the topic of how you can actually respond to digital',
 'description as a company And actually there are three different response strategies that companies can apply The \nfirst one is to team up which means that you engage in collaboration or even acquire a disruptor a Second strategy',
 "can be to fight back So that you really decide okay, we are disrupted by another company now. Let's fight back and \ncreate our own disrupted business model and Finally, and this is quite a special strategy you can also go for the",
 "escape option and I will later on explain what that exactly means So let's first focus on the team up strategy So this is \na strategy where you decide as a company to actually collaborate with a potential Disruptor of your business model",
 'and here is just one example. So 

In [12]:
# Load PDF into docs
loaders = [
    PyPDFLoader("Session 1-5.pdf"),
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

# Split docs into chunks
    
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

splits = text_splitter.split_documents(docs)

len(splits)

6

In [13]:
# create embeddings for all the chunks

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


C:\Users\dries.faems\AppData\Local\Temp\ipykernel_33508\505560370.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 990.38it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [17]:
# store embeddings in vectordatabase
# pip install faiss-cpu

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splits, embeddings)

In [18]:
vectorstore

In [20]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
)

In [23]:
# setting up the retrieval function using modern LCEL approach
# pip install langchain

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create a simple RAG chain
template = """Answer the question based only on the following context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever()

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
# execute the chain

query = "What are strategies to deal with digital disruption?"

result = chain.invoke(query)

In [25]:
print(result)

According to the context, there are three strategies to deal with digital disruption:

1. **Team up**: Collaborate or acquire a disruptor. This can be done through strategic arrangements, such as Adidas partnering with Zalando, or acquiring the disruptor, like Facebook acquiring WhatsApp.

2. **Fight back**: Create your own disruptive business model. An example is the Weather Channel, which transformed its traditional TV channel into a digital platform and now collaborates with companies like L'Oreal and Mercedes.

3. **Escape**: This strategy involves not trying to collaborate with or fight the disruptor, but rather finding alternative ways to deal with the disruption. The context does not provide a detailed example of this strategy, but mentions that it will be explained later.


In [26]:
result

"According to the context, there are three strategies to deal with digital disruption:\n\n1. **Team up**: Collaborate or acquire a disruptor. This can be done through strategic arrangements, such as Adidas partnering with Zalando, or acquiring the disruptor, like Facebook acquiring WhatsApp.\n\n2. **Fight back**: Create your own disruptive business model. An example is the Weather Channel, which transformed its traditional TV channel into a digital platform and now collaborates with companies like L'Oreal and Mercedes.\n\n3. **Escape**: This strategy involves not trying to collaborate with or fight the disruptor, but rather finding alternative ways to deal with the disruption. The context does not provide a detailed example of this strategy, but mentions that it will be explained later."